In [0]:
%sql
DROP MATERIALIZED VIEW IF EXISTS workspace.default.count_mart;
drop materialized view if exists workspace.default.customer_household_inc;
drop materialized view if exists workspace.default.join_cust_mrkt_unit_inc;
drop materialized view if exists workspace.default.join_cust_mrkt_unit_inc;
drop materialized view if exists workspace.default.my_role_a_data;
drop materialized view if exists workspace.default.my_role_b_data; 

drop materialized view if exists workspace.default.join_cust_mrkt_unit;
drop materialized view if exists workspace.default.my_role_c_data;
drop materialized view if exists workspace.default.my_role_d_data; 






In [0]:
# Fake data generation for testing joins between CUST and MRKT_UNIT tables
# CUST has many-to-one relationship with MRKT_UNIT (multiple customers per household)
from pyspark.sql.functions import rand, when, floor, concat, lpad, col as spark_col, lit, date_add, to_date, expr, current_timestamp, max as spark_max
import string

# Define the character pool and repeat it to make a long string
char_pool = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
long_char_string = char_pool * 20  # 1240 characters


def get_next_id_start(table_name, id_column):
    """Return the next starting key for a table, or 0 when table is missing/empty."""
    if spark.catalog.tableExists(table_name):
        max_id = spark.table(table_name).agg(spark_max(id_column).alias("max_id")).collect()[0]["max_id"]
        if max_id is not None:
            print(f"Found table {table_name}; starting {id_column} at {max_id + 1}")
            return int(max_id) + 1
        print(f"Table {table_name} exists but is empty; starting {id_column} at 0")
        return 0

    print(f"Table {table_name} does not exist; starting {id_column} at 0")
    return 0


mrkt_unit_table = "kraft_testing.test_schema.mrkt_unit_inc"
cust_table = "kraft_testing.test_schema.cust_inc"

mrkt_unit_start_id = get_next_id_start(mrkt_unit_table, "MRKT_UNIT_BUSN_ID")
cust_start_id = get_next_id_start(cust_table, "CUST_BUSN_ID")

# ============================================================================
# CREATE MRKT_UNIT (HOUSEHOLD) TABLE - The "one" side of the relationship
# ============================================================================
print("=== CREATING MRKT_UNIT (HOUSEHOLD) TABLE ===")

num_mrkt_units = 10  # 1.5 billion households
num_iterations_mrkt = 1  # Number of column sets for MRKT_UNIT

# Create base MRKT_UNIT dataframe
df_mrkt_unit = spark.range(num_mrkt_units).withColumn(
    "MRKT_UNIT_BUSN_ID",
    (spark_col("id") + lit(mrkt_unit_start_id)).cast("long")
).drop("id")

# Generate random date offset (0-59 days) for date variety
df_mrkt_unit = df_mrkt_unit.withColumn("date_offset",
    floor(rand() * 60).cast("int")
).withColumn("TM_ID",
    date_add(lit("2024-10-20"), spark_col("date_offset"))
)

# Add household-specific columns
df_mrkt_unit = df_mrkt_unit.withColumn("HOUSEHOLD_SIZE",
    floor(rand() * 6 + 1).cast("int")  # 1-6 members
).withColumn("HOUSEHOLD_INCOME_TIER",
    floor(rand() * 5 + 1).cast("int")  # 1-5 income tiers
).withColumn("HOUSEHOLD_TYPE_IND",
    when(rand() < 0.6, 0).otherwise(1)  # 60%/40% distribution
)


# Drop the temporary date_offset column
df_mrkt_unit = df_mrkt_unit.drop("date_offset")

print("MRKT_UNIT DataFrame created:")
df_mrkt_unit.show(10, truncate=True)
df_mrkt_unit.printSchema()
print(f"MRKT_UNIT row count: {df_mrkt_unit.count()}")

# ============================================================================
# CREATE CUST (CUSTOMER/MEMBER) TABLE - The "many" side of the relationship
# ============================================================================
print("\n=== CREATING CUST (CUSTOMER/MEMBER) TABLE ===")

num_customers = 45  # 4.5 billion customers (multiple per household)
num_iterations_cust = 1  # Number of column sets for CUST

# Create base CUST dataframe
df_cust = spark.range(num_customers).withColumn(
    "CUST_BUSN_ID",
    (spark_col("id") + lit(cust_start_id)).cast("long")
).drop("id")

# Create many-to-one relationship: multiple customers per household
# Map customers to households (on average ~3 customers per household)
df_cust = df_cust.withColumn("MRKT_UNIT_BUSN_ID",
    (floor(rand() * num_mrkt_units) + lit(mrkt_unit_start_id)).cast("long")
)

# Generate random date offset (0-59 days) - must match MRKT_UNIT for proper joins
df_cust = df_cust.withColumn("date_offset",
    floor(rand() * 60).cast("int")
).withColumn("TM_ID",
    date_add(lit("2024-10-20"), spark_col("date_offset"))
)

# Add customer-specific columns
df_cust = df_cust.withColumn("CUST_AGE",
    floor(rand() * 80 + 18).cast("int")  # Age 18-97
).withColumn("CUST_GENDER",
    when(rand() < 0.5, 0).otherwise(1)  # 50%/50% distribution
).withColumn("ACTV_MEM_RLTN_IND_CNT",
    when(rand() < 0.7, 0).otherwise(1)  # 70%/30% distribution
).withColumn("MEMBER_TYPE",
    floor(rand() * 4 + 1).cast("int")  # 1-4 member types
)


# Drop the temporary date_offset column
df_cust = df_cust.drop("date_offset")

print("CUST DataFrame created:")
df_cust.show(10, truncate=True)
df_cust.printSchema()
print(f"CUST row count: {df_cust.count()}")

# ============================================================================
# CREATE TEMP VIEWS FOR SQL ACCESS
# ============================================================================
print("\n=== CREATING TEMP VIEWS ===")
df_mrkt_unit.createOrReplaceTempView("mrkt_unit_inc_temp")
df_cust.createOrReplaceTempView("cust_inc_temp")

print("✓ Created temp view: mrkt_unit_inc_temp")
print("✓ Created temp view: cust_inc_temp")
print("\nYou can now join these tables using:")
print("  JOIN KEY: MRKT_UNIT_BUSN_ID and TM_ID")
print("  RELATIONSHIP: CUST (many) -> MRKT_UNIT (one)")

In [0]:
%sql
-- Show available catalogs and schemas
SHOW CATALOGS;
USE CATALOG kraft_testing;
SHOW SCHEMAS IN kraft_testing;

-- Step 1: Create table if it does not exist
CREATE TABLE IF NOT EXISTS test_schema.mrkt_unit_inc AS
SELECT * FROM mrkt_unit_inc_temp;

-- Step 2: Append to table
INSERT INTO test_schema.mrkt_unit_inc
SELECT * FROM mrkt_unit_inc_temp;

-- Repeat above pattern for cust_inc 
-- Step 1: Create table if it does not exist
CREATE TABLE IF NOT EXISTS test_schema.cust_inc AS
SELECT * FROM cust_inc_temp;

-- Step 2: Append to table
INSERT INTO test_schema.cust_inc
SELECT * FROM cust_inc_temp;